# CIFAR-10 Image Classification with CNN and MobileNetV2

This notebook compares a basic CNN, an augmented CNN, and pretrained MobileNetV2 transfer learning on CIFAR-10. It includes data exploration, preprocessing, training curves, confusion matrix, sample predictions, fine-tuning, and model comparison.

**Recommended runtime:** Google Colab with **T4 GPU**. The dataset is downloaded through `tf.keras.datasets.cifar10.load_data()`; no manual dataset upload is needed.

**Important:** Accuracy depends on the runtime, library versions, training duration, and random variation. Results are not guaranteed. Use validation metrics for model selection and reserve the test set for final evaluation.


In [ ]:
# Import the libraries required for this project
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

# Set random seeds for reproducible experiments
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Display TensorFlow and GPU information
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

Load the CIFAR-10 dataset

In [ ]:
# Download and load the CIFAR-10 dataset
(x_train_full, y_train_full), (x_test, y_test) = (
    keras.datasets.cifar10.load_data()
)

# Convert labels from shape (n, 1) to shape (n,)
y_train_full = y_train_full.squeeze()
y_test = y_test.squeeze()

# Define the ten image categories
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Display dataset dimensions
print("Training images:", x_train_full.shape)
print("Training labels:", y_train_full.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

Explore and visualize the images

In [ ]:
# Display a sample of 20 images with their class labels
plt.figure(figsize=(14, 8))

for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(x_train_full[i])
    plt.title(class_names[y_train_full[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

# Count images in each category
class_counts = pd.Series(y_train_full).value_counts().sort_index()

plt.figure(figsize=(10, 5))
sns.barplot(
    x=class_names,
    y=[class_counts[i] for i in range(10)]
)
plt.xticks(rotation=45)
plt.title("CIFAR-10 Training Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.tight_layout()
plt.show()

Preprocess the data

In [ ]:
# Reserve part of the training set for validation
x_train = x_train_full[:-5000]
y_train = y_train_full[:-5000]

x_val = x_train_full[-5000:]
y_val = y_train_full[-5000:]

# Normalize pixel values from 0-255 to 0-1
x_train = x_train.astype("float32") / 255.0
x_val = x_val.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Display the resulting dataset sizes
print("Training set:", x_train.shape)
print("Validation set:", x_val.shape)
print("Test set:", x_test.shape)

Build the Basic CNN model

In [ ]:
# Build a CNN for classifying 32x32 RGB images
def build_basic_cnn():

    model = keras.Sequential([
        keras.Input(shape=(32, 32, 3)),

        # Extract image features using convolution
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation="relu"),

        # Reduce spatial dimensions
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Learn more complex visual patterns
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.30),

        # Convert feature maps into a feature vector
        layers.Flatten(),

        # Perform classification
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.40),
        layers.Dense(10, activation="softmax")
    ])

    # Configure training and evaluation
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

cnn_model = build_basic_cnn()
cnn_model.summary()

Train the CNN

In [ ]:
# Save the best model and stop when validation performance stops improving
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_cnn.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True,
        mode="max"
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the CNN model
start_time = time.perf_counter()

history_cnn = cnn_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

cnn_training_time = time.perf_counter() - start_time

# Load the best checkpoint
cnn_model = keras.models.load_model("best_cnn.keras")

# Evaluate on the untouched test set
cnn_loss, cnn_accuracy = cnn_model.evaluate(
    x_test, y_test, verbose=0
)

print(f"Test accuracy: {cnn_accuracy:.4f}")
print(f"Test accuracy percentage: {cnn_accuracy * 100:.2f}%")
print(f"Training time: {cnn_training_time:.2f} seconds")

Plot accuracy and loss

In [ ]:
# Plot training and validation accuracy
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_cnn.history["accuracy"], label="Training Accuracy")
plt.plot(history_cnn.history["val_accuracy"], label="Validation Accuracy")
plt.title("CNN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Plot training and validation loss
plt.subplot(1, 2, 2)
plt.plot(history_cnn.history["loss"], label="Training Loss")
plt.plot(history_cnn.history["val_loss"], label="Validation Loss")
plt.title("CNN Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

Evaluate each class and create a confusion matrix

In [ ]:
# Generate class predictions on the test set
y_pred_prob = cnn_model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

# Print precision, recall, F1-score and class-wise performance
print(classification_report(
    y_test,
    y_pred,
    target_names=class_names,
    digits=4
))

# Build the confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("CNN Confusion Matrix")
plt.tight_layout()
plt.show()

Predict a single image

In [ ]:
# Select one test image
image_index = 25
image = x_test[image_index]

# Predict its class
probabilities = cnn_model.predict(
    np.expand_dims(image, axis=0),
    verbose=0
)[0]

predicted_class = np.argmax(probabilities)
actual_class = y_test[image_index]

# Visualize the prediction
plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.title(
    f"Actual: {class_names[actual_class]}\n"
    f"Predicted: {class_names[predicted_class]}\n"
    f"Confidence: {probabilities[predicted_class] * 100:.2f}%"
)
plt.axis("off")
plt.show()

Improve the CNN using data augmentation

In [ ]:
# Create random image transformations for training
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1)
])

# Build an augmented CNN
def build_augmented_cnn():

    inputs = keras.Input(shape=(32, 32, 3))

    # Apply augmentation only during training
    x = data_augmentation(inputs)

    # Extract features at increasing levels of complexity
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

aug_model = build_augmented_cnn()

# Train the augmented CNN
history_aug = aug_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=35,
    batch_size=128,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            "best_augmented_cnn.keras",
            monitor="val_accuracy",
            save_best_only=True,
            mode="max"
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=6,
            restore_best_weights=True,
            mode="max"
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )
    ],
    verbose=1
)

# Evaluate the augmented model
aug_model = keras.models.load_model("best_augmented_cnn.keras")
aug_loss, aug_accuracy = aug_model.evaluate(x_test, y_test, verbose=0)

print(f"Augmented CNN test accuracy: {aug_accuracy * 100:.2f}%")

Transfer learning with MobileNetV2

In [ ]:
# Enable mixed precision for NVIDIA T4 GPU
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time

tf.keras.mixed_precision.set_global_policy("mixed_float16")

print("GPU:", tf.config.list_physical_devices("GPU"))
print("Precision policy:", tf.keras.mixed_precision.global_policy())


# Build the MobileNetV2 transfer-learning model
def build_mobilenet_model():

    # Define the input image shape
    inputs = keras.Input(shape=(32, 32, 3))

    # Resize CIFAR-10 images for the pretrained network
    x = layers.Resizing(96, 96)(inputs)

    # Convert pixel values from [0, 1] to [-1, 1]
    x = layers.Rescaling(scale=2.0, offset=-1.0)(x)

    # Apply mild augmentation during training only
    x = layers.RandomFlip("horizontal")(x)
    x = layers.RandomTranslation(0.05, 0.05)(x)

    # Load pretrained MobileNetV2 feature extractor
    base_model = keras.applications.MobileNetV2(
        input_shape=(96, 96, 3),
        include_top=False,
        weights="imagenet"
    )

    # Freeze pretrained layers for the initial training stage
    base_model.trainable = False

    x = base_model(x, training=False)

    # Convert feature maps into a feature vector
    x = layers.GlobalAveragePooling2D()(x)

    # Reduce overfitting
    x = layers.Dropout(0.3)(x)

    # Keep output in float32 for numerical stability
    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32"
    )(x)

    model = keras.Model(inputs, outputs)

    # Compile the classification head
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


# Create the model
mobile_model = build_mobilenet_model()

# Save the best initial model using validation accuracy
initial_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "mobilenet_stage1_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True,
        mode="max"
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the new classification head
start_time = time.perf_counter()

history_mobile = mobile_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=128,
    callbacks=initial_callbacks,
    verbose=1
)

initial_training_time = time.perf_counter() - start_time

# Restore and preserve the best initial model for comparison
mobile_model_initial = keras.models.load_model("mobilenet_stage1_best.keras")
mobile_model = keras.models.load_model("mobilenet_stage1_best.keras")

# Evaluate the initial model
mobile_loss, mobile_accuracy = mobile_model_initial.evaluate(
    x_test, y_test, verbose=0
)

print(f"Initial test accuracy: {mobile_accuracy * 100:.2f}%")
print(f"Initial training time: {initial_training_time:.2f} seconds")


# --------------------------------------------------
# Stage 2: Fine-tune the final pretrained layers
# --------------------------------------------------

# Locate the MobileNetV2 backbone
base_model = next(
    layer for layer in mobile_model.layers
    if isinstance(layer, keras.Model)
    and "mobilenetv2" in layer.name.lower()
)

# Unfreeze the backbone, then freeze earlier layers
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization layers frozen for stability
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Recompile with a much smaller learning rate
mobile_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Save the best fine-tuned model
fine_callbacks = [
    keras.callbacks.ModelCheckpoint(
        "mobilenet_finetuned_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True,
        mode="max"
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

# Fine-tune the model
fine_start = time.perf_counter()

history_fine = mobile_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=12,
    batch_size=128,
    callbacks=fine_callbacks,
    verbose=1
)

fine_training_time = time.perf_counter() - fine_start

# Load the best fine-tuned checkpoint
mobile_model = keras.models.load_model(
    "mobilenet_finetuned_best.keras"
)

# Final evaluation on the test set
fine_loss, fine_accuracy = mobile_model.evaluate(
    x_test, y_test, verbose=0
)

print("\nFinal MobileNetV2 Results")
print(f"Test loss: {fine_loss:.4f}")
print(f"Test accuracy: {fine_accuracy * 100:.2f}%")
print(f"Fine-tuning time: {fine_training_time:.2f} seconds")

# Save the final model
mobile_model.save("cifar10_mobilenetv2_final.keras")
print("Model saved successfully.")

Compare the models

In [ ]:
# Collect test performance of all trained models
results = pd.DataFrame({
    "Model": [
        "Basic CNN",
        "Augmented CNN",
        "MobileNetV2 (Initial)",
        "MobileNetV2 (Fine-tuned)"
    ],
    "Test Accuracy (%)": [
        cnn_accuracy * 100,
        aug_accuracy * 100,
        mobile_accuracy * 100,
        fine_accuracy * 100
    ]
})

# Sort models by test accuracy
results = results.sort_values(
    by="Test Accuracy (%)",
    ascending=False
).reset_index(drop=True)

# Display the comparison table
print("Model Performance Comparison")
display(results.round(2))

# Plot model accuracy
plt.figure(figsize=(11, 6))

ax = sns.barplot(
    data=results,
    x="Model",
    y="Test Accuracy (%)"
)

# Display accuracy values above the bars
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%", padding=3)

plt.title("CIFAR-10 Model Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Test Accuracy (%)")
plt.ylim(0, 100)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Identify the model with the highest test accuracy
best_model_name = results.iloc[0]["Model"]
best_accuracy = results.iloc[0]["Test Accuracy (%)"]

print(f"Best model: {best_model_name}")
print(f"Test accuracy: {best_accuracy:.2f}%")

Save the best model

In [ ]:
# Select the highest-scoring model from the comparison table for this experiment.
# For rigorous model selection, prefer validation accuracy and evaluate the test set only once.
model_candidates = {
    "Basic CNN": cnn_model,
    "Augmented CNN": aug_model,
    "MobileNetV2 (Initial)": mobile_model_initial,
    "MobileNetV2 (Fine-tuned)": mobile_model
}

best_name = results.iloc[0]["Model"]
best_model = model_candidates[best_name]

# Save the selected model
best_model.save("cifar10_best_model.keras")

print("Selected model:", best_name)
print("Test accuracy (%):", round(float(results.iloc[0]["Test Accuracy (%)"]), 2))
print("Saved as: cifar10_best_model.keras")

# Download the saved model when running in Google Colab
try:
    from google.colab import files
    files.download("cifar10_best_model.keras")
except ImportError:
    print("Not running in Google Colab; model is saved in the current directory.")
